In [0]:
import numpy as np
import jax
import jax.numpy as jnp

# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# SAME OUTPUT MAP AS TRAINING
# ============================================================
rho_f = DTYPE(11096.0)
V_INLET_BC = DTYPE(0.4)
T_INLET = DTYPE(560.0)

PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(300.0)
TF_OUT_SCALE  = DTYPE(150.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(15.0)

# ============================================================
# Paths
# ============================================================
theta_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/adam_pinn_params_restart.npy"
phi_path    = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi_correct.npy"
tfuel_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfuel.npy"
tfluid_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfluid.npy"

layer_sizes = [3, 64, 64, 64, 6]

# ============================================================
# Helpers
# ============================================================
def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]
    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size]
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.shape[0]:
        raise ValueError(f"Unused parameters remain: used {idx}, total {theta.shape[0]}")
    return params

def load_params(theta_path, layer_sizes):
    loaded = np.load(theta_path, allow_pickle=True)
    print("loaded type :", type(loaded))
    print("loaded dtype:", getattr(loaded, "dtype", None))
    print("loaded shape:", getattr(loaded, "shape", None))
    theta = jnp.array(loaded, dtype=DTYPE)
    params = unflatten_params(theta, layer_sizes)
    return params

def predict_on_rect_grid(params, xlo, xhi, ylo, yhi, tlo, thi, Nt, Nx, Ny):
    t_vals = np.linspace(float(tlo), float(thi), Nt)
    x_vals = np.linspace(float(xlo), float(xhi), Nx)
    y_vals = np.linspace(float(ylo), float(yhi), Ny)

    Tg, Xg, Yg = np.meshgrid(t_vals, x_vals, y_vals, indexing="ij")
    pts = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

    pred = np.array(mlp_apply(params, jnp.array(pts, dtype=DTYPE)))
    pred = pred.reshape(Nt, Nx, Ny, 6)
    return pred

def rel_l2(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a)
    b = b - np.mean(b)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def print_field_stats(name, arr):
    arr = np.asarray(arr)
    print(
        f"{name}: shape={arr.shape}, "
        f"min={arr.min():.6e}, max={arr.max():.6e}, "
        f"mean={arr.mean():.6e}, std={arr.std():.6e}"
    )

# ============================================================
# Load
# ============================================================
params = load_params(theta_path, layer_sizes)

print("\nLoaded params layers:", len(params))
for i, layer in enumerate(params):
    print(f"layer {i}: W{tuple(layer['W'].shape)}, b{tuple(layer['b'].shape)}")

phi_all = np.load(phi_path)
tfuel_all = np.load(tfuel_path)
tfluid_all = np.load(tfluid_path)

print("phi_all.shape   =", phi_all.shape)
print("tfuel_all.shape =", tfuel_all.shape)
print("tfluid_all.shape=", tfluid_all.shape)

phi_true   = phi_all[0] if phi_all.ndim == 4 else phi_all
tfuel_true = tfuel_all[0] if tfuel_all.ndim == 5 else tfuel_all
tfluid_true= tfluid_all[0] if tfluid_all.ndim == 5 else tfluid_all

Ts_true = tfuel_true[0]
Tf_true = tfluid_true[0]
p_true  = tfluid_true[1]
u_true  = tfluid_true[2]
v_true  = tfluid_true[3]

print("phi_true.shape   =", phi_true.shape)
print("tfuel_true.shape =", tfuel_true.shape)
print("tfluid_true.shape=", tfluid_true.shape)

# ============================================================
# Predict
# ============================================================
Nt_phi, Nx_phi, Ny_phi = phi_true.shape
pred_phi_all = predict_on_rect_grid(params, x_min, x_max, y_min, y_max, t_min, t_max, Nt_phi, Nx_phi, Ny_phi)
phi_pred = pred_phi_all[..., 0]

Nt_Ts, Nx_Ts, Ny_Ts = Ts_true.shape
pred_Ts_all = predict_on_rect_grid(params, x_min, Ls, y_min, y_max, t_min, t_max, Nt_Ts, Nx_Ts, Ny_Ts)
Ts_pred = pred_Ts_all[..., 1]

Nt_f, Nx_f, Ny_f = Tf_true.shape
pred_f_all = predict_on_rect_grid(params, Ls, x_max, y_min, y_max, t_min, t_max, Nt_f, Nx_f, Ny_f)
Tf_pred = pred_f_all[..., 5]
p_pred  = pred_f_all[..., 4]
u_pred  = pred_f_all[..., 2]
v_pred  = pred_f_all[..., 3]

print("\nPred shapes:")
print("phi_pred:", phi_pred.shape, "phi_true:", phi_true.shape)
print("Ts_pred :", Ts_pred.shape,  "Ts_true :", Ts_true.shape)
print("Tf_pred :", Tf_pred.shape,  "Tf_true :", Tf_true.shape)
print("u_pred  :", u_pred.shape,   "u_true  :", u_true.shape)
print("v_pred  :", v_pred.shape,   "v_true  :", v_true.shape)
print("p_pred  :", p_pred.shape,   "p_true  :", p_true.shape)

print("\n================ FIELD STATS ================\n")
print_field_stats("phi_pred", phi_pred)
print_field_stats("phi_true", phi_true)
print_field_stats("Ts_pred", Ts_pred)
print_field_stats("Ts_true", Ts_true)
print_field_stats("Tf_pred", Tf_pred)
print_field_stats("Tf_true", Tf_true)
print_field_stats("u_pred", u_pred)
print_field_stats("u_true", u_true)
print_field_stats("v_pred", v_pred)
print_field_stats("v_true", v_true)
print_field_stats("p_pred", p_pred)
print_field_stats("p_true", p_true)

print("\n================ Relative L2 Errors ================\n")
print("phi :", rel_l2(phi_pred, phi_true))
print("Ts  :", rel_l2(Ts_pred, Ts_true))
print("Tf  :", rel_l2(Tf_pred, Tf_true))
print("u   :", rel_l2(u_pred,  u_true))
print("v   :", rel_l2(v_pred,  v_true))
print("p(raw)      :", rel_l2(p_pred, p_true))
print("p(zero-mean):", rel_l2_zero_mean_pressure(p_pred, p_true))

loaded type : <class 'numpy.ndarray'>
loaded dtype: float64
loaded shape: (8966,)

Loaded params layers: 4
layer 0: W(3, 64), b(64,)
layer 1: W(64, 64), b(64,)
layer 2: W(64, 64), b(64,)
layer 3: W(64, 6), b(6,)
phi_all.shape   = (17, 20, 64)
tfuel_all.shape = (1, 2, 17, 8, 64)
tfluid_all.shape= (1, 4, 17, 12, 64)
phi_true.shape   = (17, 20, 64)
tfuel_true.shape = (2, 17, 8, 64)
tfluid_true.shape= (4, 17, 12, 64)

Pred shapes:
phi_pred: (17, 20, 64) phi_true: (17, 20, 64)
Ts_pred : (17, 8, 64) Ts_true : (17, 8, 64)
Tf_pred : (17, 12, 64) Tf_true : (17, 12, 64)
u_pred  : (17, 12, 64) u_true  : (17, 12, 64)
v_pred  : (17, 12, 64) v_true  : (17, 12, 64)
p_pred  : (17, 12, 64) p_true  : (17, 12, 64)

================ FIELD STATS ================

phi_pred: shape=(17, 20, 64), min=-1.778509e+00, max=2.743749e+00, mean=6.900454e-01, std=7.379432e-01
phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.316306e+

In [15]:
def stats(name, a):
    a = np.asarray(a)
    print(f"{name}: shape={a.shape}, min={a.min():.6e}, max={a.max():.6e}, mean={a.mean():.6e}, std={a.std():.6e}")

print("\n================ FIELD STATS ================\n")

stats("phi_true", phi_true)
stats("Ts_pred", Ts_pred)
stats("Ts_true", Ts_true)
stats("Tf_pred", Tf_pred)
stats("Tf_true", Tf_true)
stats("u_pred", u_pred)
stats("u_true", u_true)
stats("v_pred", v_pred)
stats("v_true", v_true)
stats("p_pred", p_pred)
stats("p_true", p_true)

print("\n================ VELOCITY CHECKS ================\n")
print("u vs u   :", rel_l2(u_pred, u_true))
print("u vs v   :", rel_l2(u_pred, v_true))
print("u vs -u  :", rel_l2(u_pred, -u_true))
print("v vs v   :", rel_l2(v_pred, v_true))
print("v vs u   :", rel_l2(v_pred, u_true))
print("v vs -v  :", rel_l2(v_pred, -v_true))

print("\n================ PRESSURE CHECKS ================\n")
p_pred_c = p_pred - np.mean(p_pred)
p_true_c = p_true - np.mean(p_true)
print("p raw       :", rel_l2(p_pred, p_true))
print("p zero-mean :", rel_l2(p_pred_c, p_true_c))


================ FIELD STATS ================

phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.371543e+02, max=6.896059e+02, mean=6.088422e+02, std=3.511110e+01
Ts_true: shape=(17, 8, 64), min=5.600000e+02, max=9.154519e+02, mean=6.452232e+02, std=6.677291e+01
Tf_pred: shape=(17, 12, 64), min=5.543454e+02, max=6.522215e+02, mean=5.844043e+02, std=2.062213e+01
Tf_true: shape=(17, 12, 64), min=5.597972e+02, max=7.193466e+02, mean=5.737398e+02, std=2.385339e+01
u_pred: shape=(17, 12, 64), min=-1.785794e-01, max=8.709714e-01, mean=8.254723e-02, std=2.228065e-01
u_true: shape=(17, 12, 64), min=-2.153843e-03, max=1.000000e+00, mean=5.882341e-02, std=2.352942e-01
v_pred: shape=(17, 12, 64), min=-5.086100e-02, max=4.328845e-01, mean=9.029770e-02, std=1.125521e-01
v_true: shape=(17, 12, 64), min=1.000000e-12, max=4.380438e-01, mean=3.764713e-01, std=9.416659e-02
p_pred: shape=(17, 12, 64), min=-3.004108e+00

In [16]:
import numpy as np

Lf = 0.0114
Ly = 0.75
nx_f = u_true.shape[1]
ny = u_true.shape[2]

dx = Lf / nx_f
dy = Ly / ny

def stats(name, a):
    print(
        f"{name}: mean abs={np.mean(np.abs(a)):.6e}, "
        f"max abs={np.max(np.abs(a)):.6e}, "
        f"rms={np.sqrt(np.mean(a*a)):.6e}"
    )

# --------------------------------------------------
# 1) Interface (fluid left boundary x = Ls side)
# --------------------------------------------------
u_if = u_true[:, 0, :]
v_if = v_true[:, 0, :]

stats("u_if", u_if)
stats("v_if", v_if)

# --------------------------------------------------
# 2) Right boundary x-derivatives
#    cell-center finite difference diagnostic
# --------------------------------------------------
vx_right  = (v_true[:, -1, :]  - v_true[:, -2, :])  / dx
px_right  = (p_true[:, -1, :]  - p_true[:, -2, :])  / dx
Tfx_right = (Tf_true[:, -1, :] - Tf_true[:, -2, :]) / dx
ux_right  = (u_true[:, -1, :]  - u_true[:, -2, :])  / dx

stats("vx_right", vx_right)
stats("px_right", px_right)
stats("Tfx_right", Tfx_right)
stats("ux_right", ux_right)

# also value of u on right itself
u_right = u_true[:, -1, :]
stats("u_right", u_right)

# --------------------------------------------------
# 3) Outlet top y-derivatives
# --------------------------------------------------
uy_top  = (u_true[:, :, -1]  - u_true[:, :, -2])  / dy
vy_top  = (v_true[:, :, -1]  - v_true[:, :, -2])  / dy
Tfy_top = (Tf_true[:, :, -1] - Tf_true[:, :, -2]) / dy

stats("uy_top", uy_top)
stats("vy_top", vy_top)
stats("Tfy_top", Tfy_top)

# top outlet pressure value itself
p_top = p_true[:, :, -1]
stats("p_top", p_top)

# --------------------------------------------------
# 4) Optional: look time-slice by time-slice
# --------------------------------------------------
for k in [0, len(u_true)//2, len(u_true)-1]:
    print(f"\nTime index {k}")
    print("u_if mean abs:", np.mean(np.abs(u_if[k])))
    print("v_if mean abs:", np.mean(np.abs(v_if[k])))
    print("u_right mean abs:", np.mean(np.abs(u_right[k])))
    print("vx_right mean abs:", np.mean(np.abs(vx_right[k])))
    print("px_right mean abs:", np.mean(np.abs(px_right[k])))
    print("Tfx_right mean abs:", np.mean(np.abs(Tfx_right[k])))
    print("uy_top mean abs:", np.mean(np.abs(uy_top[k])))
    print("vy_top mean abs:", np.mean(np.abs(vy_top[k])))
    print("Tfy_top mean abs:", np.mean(np.abs(Tfy_top[k])))
    print("p_top mean abs:", np.mean(np.abs(p_top[k])))

u_if: mean abs=5.883133e-02, max abs=1.000000e+00, rms=2.425356e-01
v_if: mean abs=3.764690e-01, max abs=4.299048e-01, rms=3.880792e-01
vx_right: mean abs=5.890303e-01, max abs=1.086036e+01, rms=1.316545e+00
px_right: mean abs=1.433571e+00, max abs=1.560223e+02, rms=1.133412e+01
Tfx_right: mean abs=1.991554e+02, max abs=1.210774e+03, rms=3.522044e+02
ux_right: mean abs=1.633805e-02, max abs=8.265961e-01, rms=5.754749e-02
u_right: mean abs=5.883472e-02, max abs=1.000000e+00, rms=2.425356e-01
uy_top: mean abs=7.615197e-03, max abs=1.828141e-01, rms=3.279862e-02
vy_top: mean abs=8.815635e-03, max abs=3.037577e-01, rms=3.829400e-02
Tfy_top: mean abs=4.466077e+01, max abs=5.895366e+02, rms=1.127513e+02
p_top: mean abs=7.212336e-03, max abs=2.341059e-01, rms=3.125286e-02

Time index 0
u_if mean abs: 1.0
v_if mean abs: 1e-12
u_right mean abs: 1.0
vx_right mean abs: 0.0
px_right mean abs: 0.0
Tfx_right mean abs: 0.0
uy_top mean abs: 0.0
vy_top mean abs: 0.0
Tfy_top mean abs: 0.0
p_top mean abs

In [17]:
import numpy as np

phi = np.load("/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi.npy")[0]
# phi shape should be (17, 20, 64)

Ly = 0.75
y = np.linspace(0.0, Ly, phi.shape[2])
phi_ic_expected = 2.0 * np.cos(3.14 * (y - 0.375) / 0.75)

# compare first stored time slice against expected IC
phi_t0 = phi[0]            # shape (20,64)

# since IC depends only on y, repeat across x
phi_ic_grid = np.tile(phi_ic_expected[None, :], (phi.shape[1], 1))

err = phi_t0 - phi_ic_grid

print("phi_t0 vs expected IC")
print("mean abs:", np.mean(np.abs(err)))
print("max abs :", np.max(np.abs(err)))
print("phi_t0 min/max:", phi_t0.min(), phi_t0.max())
print("expected min/max:", phi_ic_grid.min(), phi_ic_grid.max())

phi_t0 vs expected IC
mean abs: 1.253745778718938
max abs : 1.999378994095175
phi_t0 min/max: 0.0 0.0
expected min/max: 0.0015926534214665267 1.999378994095175


ModuleNotFoundError: No module named 'netCDF4'

In [0]:
print("phi_true shape:", phi_true.shape)
print("phi_true min :", np.min(phi_true))
print("phi_true max :", np.max(phi_true))
print("phi_true mean:", np.mean(phi_true))
print("phi_true std :", np.std(phi_true))

print("phi_pred min :", np.min(phi_pred))
print("phi_pred max :", np.max(phi_pred))

rel_phi = np.linalg.norm(phi_pred - phi_true) / np.linalg.norm(phi_true)
print("phi relative L2:", rel_phi)

phi_true shape: (17, 20, 64)
phi_true min : 0.019286873502372324
phi_true max : 10.483047280042607
phi_true mean: 2.0318494436571943
phi_true std : 1.688021063367896
phi_pred min : -0.4731496666171897
phi_pred max : 3.279056286493864
phi relative L2: 0.626565313235716


: 